In [22]:
import pandas as pd

# 1. Đọc file JSON bằng thư viện json tiêu chuẩn
with open('dataset/services.json', 'r', encoding='utf-8') as f:
    json_data = json.load(f)

# 2. Làm phẳng dữ liệu và chuyển thành DataFrame
# (Hàm này sẽ tự động xử lý các mảng có độ dài khác nhau)
df_json = pd.json_normalize(json_data)
# Đọc file JSONL (thêm tham số lines=True)
df_jsonl = pd.read_json('dataset/alerts_sample.jsonl', lines=True)
print(df_json.head())

print(df_jsonl.head())

                                            services  \
0  [{'name': 'edge-lb', 'tier': 'edge', 'team': '...   

                                              stores  \
0  [{'name': 'payments-db', 'type': 'postgres', '...   

                                               edges  \
0  [{'from': 'edge-lb', 'to': 'auth-svc', 'type':...   

                                      topology_notes  \
0  [edge-lb là entry point — alert tại edge-lb th...   

                                      _meta.scenario _meta.schema_version  \
0  Synthetic e-commerce platform 'GeekShop'. 10 s...                  1.0   

                                 _meta.generated_for  \
0  AIOps W2 group lab (Thu Jun 11 - Fri Jun 12, 2...   

                                   _meta.nab_mapping  
0  Mỗi service có 1 NAB stream gán cố định cho me...  
       id                    ts       service                         metric  \
0  a-0001  2026-06-12T09:42:01Z   payment-svc  db_connection_pool_used_ratio   
1  a-0002 

In [23]:
import pandas as pd

# 1. Đọc file JSON bằng thư viện json tiêu chuẩn
with open('dataset/services.json', 'r', encoding='utf-8') as f:
    json_data = json.load(f)

# 2. Làm phẳng dữ liệu và chuyển thành DataFrame
# (Hàm này sẽ tự động xử lý các mảng có độ dài khác nhau)
df_json = pd.json_normalize(json_data)
# Đọc file JSONL (thêm tham số lines=True)
df_jsonl = pd.read_json('dataset/alerts_sample.jsonl', lines=True)
print(df_json.head())

print(df_jsonl.head())

                                            services  \
0  [{'name': 'edge-lb', 'tier': 'edge', 'team': '...   

                                              stores  \
0  [{'name': 'payments-db', 'type': 'postgres', '...   

                                               edges  \
0  [{'from': 'edge-lb', 'to': 'auth-svc', 'type':...   

                                      topology_notes  \
0  [edge-lb là entry point — alert tại edge-lb th...   

                                      _meta.scenario _meta.schema_version  \
0  Synthetic e-commerce platform 'GeekShop'. 10 s...                  1.0   

                                 _meta.generated_for  \
0  AIOps W2 group lab (Thu Jun 11 - Fri Jun 12, 2...   

                                   _meta.nab_mapping  
0  Mỗi service có 1 NAB stream gán cố định cho me...  
       id                    ts       service                         metric  \
0  a-0001  2026-06-12T09:42:01Z   payment-svc  db_connection_pool_used_ratio   
1  a-0002 

Fingerprint 

In [24]:
import pandas as pd

# Định nghĩa hàm của bạn
def fingerprint(alert: dict) -> str:
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"

# Chạy trên cả bảng bằng cách dùng .apply() với tham số axis=1 (chạy theo từng dòng)
df_jsonl['fingerprint'] = df_jsonl.apply(fingerprint, axis=1)

# In thử bảng mới để kiểm tra kết quả
print(df_jsonl[['fingerprint']].head())

                                      fingerprint
0  payment-svc|db_connection_pool_used_ratio|warn
1  payment-svc|db_connection_pool_used_ratio|crit
2                 payment-svc|latency_p99_ms|crit
3                     payment-svc|error_rate|warn
4                checkout-svc|latency_p99_ms|warn


Fingerprint + state

In [25]:
class Deduper:
    def __init__(self):
        self.store: dict[str, dict] = {}

    def push(self, alert: dict) -> str:
        fp = fingerprint(alert)
        if fp not in self.store:
            self.store[fp] = {
                'cluster_id': fp,
                'count': 1,
                'first_seen': alert['ts'],
                'last_seen': alert['ts'],
                'alerts': [alert['id']],
            }
        else:
            c = self.store[fp]
            c['count'] += 1
            c['last_seen'] = alert['ts']
            c['alerts'].append(alert['id'])
        return fp
deduper = Deduper()
for _, alert in df_jsonl.iterrows():
    deduper.push(alert) 
df_clusters = pd.DataFrame(deduper.store.values())
top_5_clusters = df_clusters.sort_values(by='count', ascending=False).head(5)
for _, cluster in top_5_clusters.iterrows():
    print(f"Cluster ID: {cluster['cluster_id']}, "
          f"Count: {cluster['count']}, "
          f"First Seen: {cluster['first_seen']}, "
          f"Last Seen: {cluster['last_seen']}, "
          f"Alert IDs: {cluster['alerts']}\n")
    

Cluster ID: payment-svc|latency_p99_ms|crit, Count: 3, First Seen: 2026-06-12T09:42:22Z, Last Seen: 2026-06-12T09:46:01Z, Alert IDs: ['a-0003', 'a-0008', 'a-0015']

Cluster ID: payment-svc|db_connection_pool_used_ratio|crit, Count: 2, First Seen: 2026-06-12T09:42:18Z, Last Seen: 2026-06-12T09:44:02Z, Alert IDs: ['a-0002', 'a-0011']

Cluster ID: payment-svc|db_connection_pool_used_ratio|warn, Count: 1, First Seen: 2026-06-12T09:42:01Z, Last Seen: 2026-06-12T09:42:01Z, Alert IDs: ['a-0001']

Cluster ID: payment-svc|error_rate|warn, Count: 1, First Seen: 2026-06-12T09:42:30Z, Last Seen: 2026-06-12T09:42:30Z, Alert IDs: ['a-0004']

Cluster ID: checkout-svc|latency_p99_ms|warn, Count: 1, First Seen: 2026-06-12T09:42:45Z, Last Seen: 2026-06-12T09:42:45Z, Alert IDs: ['a-0005']



Session window

In [26]:
from dateutil.parser import parse
def session_groups(alerts: list[dict], gap_sec: int = 120) -> list[list[dict]]:
    """Mỗi group là 1 'session'. Session ngắt khi gap > gap_sec giây."""
    if not alerts:
        return []
    sorted_alerts = sorted(alerts, key=lambda a: a['ts'])
    groups = [[sorted_alerts[0]]]
    for alert in sorted_alerts[1:]:
        last_ts = parse(groups[-1][-1]['ts'])
        if (parse(alert['ts']) - last_ts).total_seconds() <= gap_sec:
            groups[-1].append(alert)
        else:
            groups.append([alert])
    return groups
all_alerts = df_jsonl.to_dict(orient='records')
global_sessions = session_groups(all_alerts, gap_sec=120)
print(f"Total sessions: {len(global_sessions)}")
for i, session in enumerate(global_sessions[:5]):
    print(f"Session {i+1}:")
    for alert in session:
        print(f"  Alert ID: {alert['id']}, Timestamp: {alert['ts']}")
    print()
    


Total sessions: 2
Session 1:
  Alert ID: a-0001, Timestamp: 2026-06-12T09:42:01Z
  Alert ID: a-0002, Timestamp: 2026-06-12T09:42:18Z
  Alert ID: a-0003, Timestamp: 2026-06-12T09:42:22Z
  Alert ID: a-0004, Timestamp: 2026-06-12T09:42:30Z
  Alert ID: a-0005, Timestamp: 2026-06-12T09:42:45Z
  Alert ID: a-0006, Timestamp: 2026-06-12T09:43:01Z
  Alert ID: a-0007, Timestamp: 2026-06-12T09:43:15Z
  Alert ID: a-0009, Timestamp: 2026-06-12T09:43:32Z
  Alert ID: a-0010, Timestamp: 2026-06-12T09:43:50Z
  Alert ID: a-0011, Timestamp: 2026-06-12T09:44:02Z
  Alert ID: a-0012, Timestamp: 2026-06-12T09:44:30Z
  Alert ID: a-0013, Timestamp: 2026-06-12T09:45:10Z
  Alert ID: a-0014, Timestamp: 2026-06-12T09:45:42Z
  Alert ID: a-0015, Timestamp: 2026-06-12T09:46:01Z
  Alert ID: a-0016, Timestamp: 2026-06-12T09:46:50Z
  Alert ID: a-0017, Timestamp: 2026-06-12T09:47:12Z
  Alert ID: a-0018, Timestamp: 2026-06-12T09:48:01Z
  Alert ID: a-0019, Timestamp: 2026-06-12T09:48:25Z
  Alert ID: a-0020, Timestamp: 2026

Tạo graph 

In [34]:
import networkx as nx
G = nx.DiGraph()
for edge in json_data['edges']:
    # Thêm cạnh nối giữa 2 service, kèm theo thuộc tính loại kết nối (http, kafka,...)
    G.add_edge(edge['from'], edge['to'], type=edge['type'])
print(f"📌 TỔNG SỐ SERVICES ({G.number_of_nodes()}):")
# G.nodes trả về danh sách các node, chúng ta sắp xếp lại cho dễ nhìn
for node in sorted(G.nodes):
    print(f"  - {node}")

print("\n" + "="*50 + "\n")
print(f"🔌 TỔNG SỐ CONNECTIONS ({G.number_of_edges()}):")
for u, v, d in G.edges(data=True):
    connection_type = d.get('type', 'unknown')
    print(f"  [{u}] ----({connection_type})----> [{v}]")


📌 TỔNG SỐ SERVICES (14):
  - auth-svc
  - cart-redis
  - cart-svc
  - catalog-db
  - catalog-svc
  - checkout-svc
  - edge-lb
  - inventory-svc
  - kafka-events
  - notification-svc
  - payment-svc
  - payments-db
  - recommender-svc
  - search-svc


🔌 TỔNG SỐ CONNECTIONS (17):
  [edge-lb] ----(http)----> [auth-svc]
  [edge-lb] ----(http)----> [catalog-svc]
  [edge-lb] ----(http)----> [search-svc]
  [edge-lb] ----(http)----> [checkout-svc]
  [catalog-svc] ----(postgres)----> [catalog-db]
  [catalog-svc] ----(http)----> [recommender-svc]
  [search-svc] ----(postgres)----> [catalog-db]
  [checkout-svc] ----(http)----> [cart-svc]
  [checkout-svc] ----(http)----> [payment-svc]
  [checkout-svc] ----(http)----> [inventory-svc]
  [checkout-svc] ----(kafka)----> [notification-svc]
  [cart-svc] ----(redis)----> [cart-redis]
  [cart-svc] ----(http)----> [catalog-svc]
  [payment-svc] ----(postgres)----> [payments-db]
  [inventory-svc] ----(postgres)----> [catalog-db]
  [notification-svc] ----(kaf

topology groups

In [31]:
import networkx as nx
from collections import defaultdict

def topology_group(alerts, graph, max_hop=2):
    """Gom alert có service cách nhau ≤ max_hop trên graph."""
    undirected = graph.to_undirected()
    by_service = defaultdict(list)
    for a in alerts:
        by_service[a['service']].append(a)

    services = list(by_service.keys())
    parent = {s: s for s in services}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for i, s1 in enumerate(services):
        for s2 in services[i+1:]:
            try:
                if nx.shortest_path_length(undirected, s1, s2) <= max_hop:
                    parent[find(s1)] = find(s2)
            except nx.NetworkXNoPath:
                pass

    groups = defaultdict(list)
    for s in services:
        groups[find(s)].extend(by_service[s])
    return list(groups.values())
topology_groups = topology_group(all_alerts, G, max_hop=2)    
print(f"Total topology groups: {len(topology_groups)}")
for i, group in enumerate(topology_groups[:5]):
    print(f"Topology Group {i+1}:")
    for alert in group:
        print(f"  Alert ID: {alert['id']}, Service: {alert['service']}, Timestamp: {alert['ts']}")
    print()


Total topology groups: 1
Topology Group 1:
  Alert ID: a-0001, Service: payment-svc, Timestamp: 2026-06-12T09:42:01Z
  Alert ID: a-0002, Service: payment-svc, Timestamp: 2026-06-12T09:42:18Z
  Alert ID: a-0003, Service: payment-svc, Timestamp: 2026-06-12T09:42:22Z
  Alert ID: a-0004, Service: payment-svc, Timestamp: 2026-06-12T09:42:30Z
  Alert ID: a-0008, Service: payment-svc, Timestamp: 2026-07-12T09:43:18Z
  Alert ID: a-0011, Service: payment-svc, Timestamp: 2026-06-12T09:44:02Z
  Alert ID: a-0015, Service: payment-svc, Timestamp: 2026-06-12T09:46:01Z
  Alert ID: a-0018, Service: payment-svc, Timestamp: 2026-06-12T09:48:01Z
  Alert ID: a-0005, Service: checkout-svc, Timestamp: 2026-06-12T09:42:45Z
  Alert ID: a-0006, Service: checkout-svc, Timestamp: 2026-06-12T09:43:01Z
  Alert ID: a-0012, Service: checkout-svc, Timestamp: 2026-06-12T09:44:30Z
  Alert ID: a-0017, Service: checkout-svc, Timestamp: 2026-06-12T09:47:12Z
  Alert ID: a-0007, Service: edge-lb, Timestamp: 2026-06-12T09:43

In [37]:
def json_encoder(obj):
    # Nếu gặp DataFrame, chuyển về dạng list of dicts
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient='records')
    # Nếu gặp Series, chuyển về dạng dict
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    # Nếu gặp set (tập hợp), chuyển về list
    if isinstance(obj, set):
        return list(obj)
    raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")

Time-Window + Topology

In [ ]:
import json


def correlate(alerts, graph, gap_sec=120, max_hop=2):
    sessions = session_groups(alerts, gap_sec=gap_sec)
    clusters = []
    for s_idx, session_alerts in enumerate(sessions):
        for g_idx, group in enumerate(topology_group(session_alerts, graph, max_hop)):
            clusters.append({
                'cluster_id': f'c-{s_idx:03d}-{g_idx:03d}',
                'alert_count': len(group),
                'services': sorted({a['service'] for a in group}),
                'time_range': [min(a['ts'] for a in group), max(a['ts'] for a in group)],
                'max_severity': max(a['severity'] for a in group),
                'alert_ids': [a['id'] for a in group],
                'fingerprints': sorted(list({fingerprint(a) for a in group})),
            })
    return clusters

clusters = correlate(all_alerts, G, gap_sec=120, max_hop=2)

input_count = len(all_alerts)
output_count = len(clusters)

reduction_ratio = 1 - output_count / input_count if input_count > 0 else 0.0

output_data = {
    "input_alerts": input_count,
    "output_clusters": output_count,
    "reduction_ratio": reduction_ratio,
    "clusters": clusters
}

with open('results/cluster_summary.json', 'w', encoding='utf-8') as json_file:
    json.dump(output_data, json_file, ensure_ascii=False, indent=2, default=json_encoder)

print(f"Total clusters: {output_count}")
for cluster in clusters[:5]:     
    print(f"Cluster ID: {cluster['cluster_id']}, "
          f"Alert Count: {cluster['alert_count']}, "
          f"Services: {cluster['services']}, "
          f"Time Range: {cluster['time_range']}, "
          f"Max Severity: {cluster['max_severity']}, "
          f"Fingerprints: {cluster['fingerprints']}")
    


Total clusters: 2
Cluster ID: c-000-000, Alert Count: 19, Services: ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'recommender-svc', 'search-svc'], Time Range: ['2026-06-12T09:42:01Z', '2026-06-12T09:48:30Z'], Max Severity: warn, Fingerprints: ['cart-svc|latency_p99_ms|warn', 'checkout-svc|downstream_payment_error_rate|crit', 'checkout-svc|latency_p99_ms|crit', 'checkout-svc|latency_p99_ms|warn', 'checkout-svc|request_drop_rate|crit', 'edge-lb|p99_latency_ms|crit', 'edge-lb|upstream_5xx_rate|crit', 'edge-lb|upstream_5xx_rate|warn', 'notification-svc|queue_depth|crit', 'notification-svc|queue_lag_ms|warn', 'payment-svc|db_connection_pool_used_ratio|crit', 'payment-svc|db_connection_pool_used_ratio|warn', 'payment-svc|error_rate|crit', 'payment-svc|error_rate|warn', 'payment-svc|latency_p99_ms|crit', 'recommender-svc|cpu_utilization|warn', 'search-svc|catalog_db_query_time_ms|warn']
Cluster ID: c-001-000, Alert Count: 1, Services: ['payment-svc'], Time Range: